# Citiz — 2. Normalisation et contrôles (Silver)
Runtime Python. Même environnement de dépendances que Bronze. Lakehouses dans le même workspace.
Les codes INSEE restent des chaînes. Affectation spatiale par contours, pas par centre le plus proche.
GBFS vélo/trottinette et systèmes mixtes non qualifiés exclus de la concurrence automobile.
Une source manquante n'est jamais interprétée comme une absence de transports, concurrents ou délits.
Les projections Omphale (échelle supracommunale) et l'état des chaussées ne servent pas de proxy
à la croissance locale ou aux embouteillages.


In [ ]:
BRONZE = 'Bronze'
SILVER = 'Silver'
SCHEMA = 'dbo'
LOCAL_ROOT = None
MAX_INVENTORY_AGE_DAYS = 180
MAX_MEASUREMENT_AGE_DAYS = 730


In [ ]:
import os, re, json, unicodedata, uuid, datetime as dt
import numpy as np
import pandas as pd
import pyarrow as pa
from deltalake import DeltaTable, write_deltalake
from scipy.spatial import cKDTree
RUN_ID = str(uuid.uuid4())
STATUS = []

def nu():
    import notebookutils
    return notebookutils

def root(lakehouse):
    if LOCAL_ROOT:
        return f"{LOCAL_ROOT}/{lakehouse}", None
    info = nu().lakehouse.get(lakehouse)
    if not isinstance(info, dict):
        info = json.loads(json.dumps(info, default=lambda x: x.__dict__))
    props = info.get("properties", {})
    path = props.get("abfsPath")
    if not path:
        raise RuntimeError(f"abfsPath absent pour {lakehouse}; contrôler notebookutils.lakehouse.get")
    return path.rstrip("/"), {"bearer_token": nu().credentials.getToken("storage"), "use_fabric_endpoint":"true"}

def table_path(lakehouse, name):
    p,o = root(lakehouse)
    return f"{p}/Tables/{SCHEMA + '/' if SCHEMA else ''}{name}",o

def read_table(lakehouse, name, required=False, run_id=None):
    p,o = table_path(lakehouse,name)
    try:
        t=DeltaTable(p,storage_options=o).to_pandas()
    except Exception as e:
        if required: raise RuntimeError(f"Table obligatoire illisible: {lakehouse}.{name}") from e
        print(f"Absente/illisible: {lakehouse}.{name}: {str(e)[:120]}")
        return pd.DataFrame()
    if run_id is not None:
        if '_run_id' not in t: return pd.DataFrame()
        t=t[t['_run_id']==run_id].copy()
    return t.drop(columns=['_run_id'],errors='ignore')

def write_table(lakehouse,name,df):
    df=df.reset_index(drop=True).copy()
    df['_run_id']=RUN_ID
    # Types explicites même pour colonnes entièrement nulles / tables vides.
    a=pa.Table.from_pandas(df,preserve_index=False)
    a=pa.Table.from_arrays([a[c].cast(pa.string()) if pa.types.is_null(a[c].type) else a[c]
                            for c in a.column_names],names=a.column_names)
    p,o=table_path(lakehouse,name)
    if LOCAL_ROOT: os.makedirs(os.path.dirname(p),exist_ok=True)
    write_deltalake(p,a,mode='overwrite',schema_mode='overwrite',storage_options=o)
    STATUS.append({'table':name,'lignes':len(df),'run_id':RUN_ID})
    print(f"{lakehouse}.{name}: {len(df):,} lignes")

def nnum(s):
    return pd.to_numeric(s.astype('string').str.replace('\u202f','',regex=False)
          .str.replace(' ','',regex=False).str.replace(',','.',regex=False),errors='coerce')

def textnorm(s):
    return s.fillna('').astype(str).map(lambda v: unicodedata.normalize('NFKD',v)
                    .encode('ascii','ignore').decode().lower().strip())

def codes(s):
    s=s.astype('string').str.strip().str.upper().str.replace(r'\.0$','',regex=True)
    return s.str.zfill(5).where(s.str.fullmatch(r'\d{1,5}|2[AB]\d{3}',na=False))

def arm_to_com(s):
    s=codes(s)
    return s.mask(s.str.match(r'751\d{2}',na=False),'75056').mask(s.str.match(r'132\d{2}',na=False),'13055').mask(s.str.match(r'6938[1-9]',na=False),'69123')

def assert_unique(df,keys,name):
    if df[keys].isna().any().any() or df.duplicated(keys).any():
        raise ValueError(f"Clé nulle ou dupliquée: {name} {keys}")

def valid_points(df):
    d=df.copy()
    d['lat']=nnum(d['lat']); d['lon']=nnum(d['lon'])
    return d[d.lat.between(-90,90)&d.lon.between(-180,180)&~((d.lat==0)&(d.lon==0))].copy()

def xyz(lon,lat):
    a,b=np.radians(np.asarray(lon,dtype=float)),np.radians(np.asarray(lat,dtype=float))
    return np.column_stack([np.cos(b)*np.cos(a),np.cos(b)*np.sin(a),np.sin(b)])

def nearest(q,refs):
    if refs.empty: return np.full(len(q),-1,dtype=int),np.full(len(q),np.nan)
    d,i=cKDTree(xyz(refs.lon,refs.lat)).query(xyz(q.lon,q.lat),k=1)
    return i,2*6371008.8*np.arcsin(np.clip(d/2,0,1))

def radius_sum(q,refs,radius,value=None):
    if refs.empty: return np.zeros(len(q))
    tree=cKDTree(xyz(refs.lon,refs.lat)); chord=2*np.sin(radius/(2*6371008.8))
    result=[]
    for start in range(0,len(q),1000):
        hits=tree.query_ball_point(xyz(q.lon.iloc[start:start+1000],q.lat.iloc[start:start+1000]),chord)
        result.extend([float(refs.iloc[h][value].sum()) if value else len(h) for h in hits])
    return np.array(result)


In [ ]:
run=read_table(BRONZE,'pipeline_run',required=True)
if len(run)!=1 or run.iloc[0]['status']!='complete': raise ValueError('Run Bronze non validé')
RUN_ID=run.iloc[0]['run_id']
def B(name,required=False): return read_table(BRONZE,name,required,RUN_ID)
com=B('communes',True).rename(columns={'code':'commune_code'})
com['commune_code']=codes(com.commune_code)
com=valid_points(com); assert_unique(com,['commune_code'],'communes')
# Jointure point-dans-polygone. Frontières: choix déterministe, jamais un centre arbitraire.
from shapely.geometry import shape, Point
from shapely.strtree import STRtree
geoms=[]; geocodes=[]
for row in com.itertuples():
    g=json.loads(row.contour_json)
    if g:
        geom=shape(g)
        if not geom.is_valid: geom=geom.buffer(0)
        if not geom.is_empty: geoms.append(geom); geocodes.append(row.commune_code)
if not geoms: raise ValueError('Contours communaux absents')
tree=STRtree(geoms)
def assign(df):
    df=valid_points(df); assigned=[]
    for x,y in zip(df.lon,df.lat):
        p=Point(x,y)
        hits=tree.query(p,predicate='intersects')
        assigned.append(sorted(geocodes[int(i)] for i in hits)[0] if len(hits) else None)
    df['commune_code']=assigned
    return df
# Annuaire dimensionnel; population_api jamais utilisée comme millésime statistique caché.
write_table(SILVER,'dim_commune',com)


In [ ]:
# Population historique: colonnes strictes, pas d'interprétation automatique de n'importe quel chiffre.
p,o=root(BRONZE)
if LOCAL_ROOT:
    names=os.listdir(f"{p}/Tables/{SCHEMA + '/' if SCHEMA else ''}")
else:
    names=[x.name.rstrip('/') for x in nu().fs.ls(f"{p}/Tables/{SCHEMA + '/' if SCHEMA else ''}")]
frames=[]
for name in names:
    if not name.startswith('population_historique_'): continue
    d=B(name)
    if d.empty: continue
    cc=next((c for c in d if c in ['codgeo','com','code','code_insee','depcom'] or c.startswith('codgeo_')),None)
    if cc is None: continue
    yc={c:int(re.search(r'(18\d{2}|19\d{2}|20\d{2})',c).group()) for c in d
         if re.fullmatch(r'(?:p|pop|population|pm|pmun|psdc|ptot)?_?(?:18|19|20)\d{2}(?:_pop)?',c)}
    if not yc: continue
    t=d.melt(id_vars=cc,value_vars=list(yc),var_name='variable',value_name='population')
    t['annee']=t.variable.map(yc); t['commune_code']=codes(t[cc]); t['population']=nnum(t.population)
    frames.append(t[['commune_code','annee','population']])
if not frames: raise ValueError('Historique INSEE: adapter explicitement les noms de colonnes')
hist=pd.concat(frames,ignore_index=True).dropna()
hist=hist[hist.commune_code.isin(com.commune_code)&(hist.population>=0)]
missing=com[~com.commune_code.isin(hist.commune_code)][['commune_code']].copy()
missing['motif']='code actuel sans historique joint: vérifier géographie/millésime'
write_table(SILVER,'qualite_geographie',missing)
hist=hist.drop_duplicates()
assert_unique(hist,['commune_code','annee'],'population historique')
write_table(SILVER,'fact_population',hist)

g=B('carreaux_population')
if len(g):
    g=assign(g); g=g[g.commune_code.notna()].copy()
    assert_unique(g,['carreau_id'],'carreaux')
    if len(g): write_table(SILVER,'fact_carreau_population',g)

l=B('logement_2022')
if len(l):
    required=['codgeo','p22_rp','p22_rp_voit1','p22_rp_voit2p']
    if not set(required)<=set(l): raise ValueError('Logement: schéma INSEE inattendu')
    t=l[required].copy(); t['commune_code']=codes(t.codgeo)
    for c in required[1:]: t[c]=nnum(t[c])
    t=t[t.commune_code.isin(com.commune_code)&(t.p22_rp>0)].copy()
    t['menages']=t.p22_rp; t['menages_sans_voiture']=t.p22_rp-t.p22_rp_voit1-t.p22_rp_voit2p
    t['part_sans_voiture']=t.menages_sans_voiture/t.menages
    if (~t.part_sans_voiture.between(0,1)).any(): raise ValueError('Ratio motorisation invalide')
    t['annee']=2022
    assert_unique(t,['commune_code'],'ménages')
    write_table(SILVER,'fact_motorisation',t[['commune_code','annee','menages','menages_sans_voiture','part_sans_voiture']])


In [ ]:
stops=B('gtfs_stops')
if len(stops):
    stops=stops.rename(columns={'stop_lat':'lat','stop_lon':'lon','stop_name':'nom'})
    stops=stops[stops.location_type.fillna('0').isin(['0',''])].copy()
    stops=valid_points(stops)
    stops['arret_id']=stops.feed_id.astype(str)+':'+stops.stop_id.astype(str)
    stops=stops.drop_duplicates('arret_id')
    # Garder aussi les arrêts hors frontière, utiles à une station voisine d'une commune.
    stops=assign(stops)
    write_table(SILVER,'dim_arret',stops[['arret_id','feed_id','nom','lat','lon','commune_code']])
s=B('gbfs_stations'); parts=[]; excluded=[]
if len(s):
    for row in s.to_dict('records'):
        cars=set(json.loads(row['car_type_ids'])); alltypes=set(json.loads(row['all_type_ids']))
        allowed=set(json.loads(row['station_type_ids']))
        # Car confirmé pour tout le système OU liste de types station explicite exclusivement auto.
        iscar=(bool(cars) and cars==alltypes) or (bool(allowed) and allowed<=cars)
        if iscar:
            row['station_key']=str(row['feed_id'])+':'+str(row['station_id'])
            row['date_observation']=row['snapshot_date']; row['source']=row['source_url']; parts.append(row)
        else: excluded.append({'station_id':row['station_id'],'feed_id':row['feed_id'],'motif':'type automobile non confirmé'})
m=B('input_stations_manuelles')
if len(m):
    m['is_citiz']=textnorm(m.is_citiz).isin(['true','1','oui'])
    m['station_key']='manuel:'+m.operator+':'+m.station_id
    parts.extend(m.to_dict('records'))
if parts:
    s=assign(pd.DataFrame(parts)); s['capacity']=nnum(s.capacity)
    age=(pd.Timestamp.now(tz='UTC').tz_localize(None).normalize()-pd.to_datetime(s.date_observation,errors='coerce')).dt.days
    s['observation_recente']=age.between(0,MAX_INVENTORY_AGE_DAYS)
    # Même station physique Citiz/PAN + inventaire manuel: dédoublonnage approximatif 10 m, opérateur inclus.
    s['_dedup']=textnorm(s.operator)+':'+s.lat.round(4).astype(str)+':'+s.lon.round(4).astype(str)
    s=s.sort_values(['date_observation','station_key']).drop_duplicates('_dedup',keep='last')
    cols=['station_key','operator','is_citiz','lat','lon','capacity','commune_code','date_observation','observation_recente','source']
    assert_unique(s,['station_key'],'stations auto')
    write_table(SILVER,'dim_station',s[cols])
if excluded: write_table(SILVER,'qualite_gbfs_exclus',pd.DataFrame(excluded))


In [ ]:
f=B('flux_domicile_travail_2022')
if len(f):
    weight=next((c for c in f if c.startswith('nbflux_c22')),None)
    if not weight or not {'codgeo','dclt'}<=set(f): raise ValueError('Schéma des flux INSEE inattendu')
    t=pd.DataFrame({'origine':arm_to_com(f.codgeo),'destination':arm_to_com(f.dclt),'nb_actifs':nnum(f[weight])})
    t=t.dropna(); t=t[t.nb_actifs>=0].groupby(['origine','destination'],as_index=False).nb_actifs.sum()
    # Agrégation ARM -> Paris/Lyon/Marseille AVANT calcul intra/intercommunal.
    t=t[t.origine.isin(com.commune_code)].copy(); t['annee']=2022
    write_table(SILVER,'fact_flux',t)

d=B('delinquance_communale')
if len(d):
    def col(candidates):
        return next((c for c in candidates if c in d),None)
    cc=col(['codgeo_2026','codgeo_2025','codgeo','code_commune','com'])
    if cc is None: cc=next((c for c in d if c.startswith('codgeo_')),None)
    ic=col(['indicateur','classe']); yc=col(['annee']); nc=col(['nombre','faits']); rc=col(['taux_pour_mille','taux_pour_mille_habitants','tauxpourmille','taux_pour_mille_hab'])
    if not all([cc,ic,yc,nc]): raise ValueError(f'SSMSI: colonnes inattendues {list(d)}')
    labels=textnorm(d[ic]); cat=pd.Series(pd.NA,index=d.index,dtype='string')
    cat[labels.str.contains(r'vols? de vehicules?')]='vol_vehicule'
    cat[labels.str.contains(r'vols? dans les vehicules?')]='vol_dans_vehicule'
    cat[labels.str.contains('accessoires')]='vol_accessoire'
    # Dégradations générales: contexte seulement, jamais intitulées "casse automobile".
    cat[labels.str.contains('destruction|degradation')]='degradations_generales'
    t=pd.DataFrame({'commune_code':codes(d[cc]),'annee':nnum(d[yc]),'categorie':cat,'nombre':nnum(d[nc]),
      'taux_pour_mille':nnum(d[rc]) if rc else np.nan})
    t['annee']=t.annee.where(t.annee>=100,t.annee+2000)
    if 'est_diffuse' in d:
        published=textnorm(d.est_diffuse).isin(['true','1','oui','vrai'])
        t.loc[~published,['nombre','taux_pour_mille']]=np.nan
    t=t[t.commune_code.isin(com.commune_code)&t.categorie.notna()&t.annee.notna()]
    t=t.drop_duplicates()
    assert_unique(t,['commune_code','annee','categorie'],'SSMSI')
    write_table(SILVER,'fact_delinquance',t)


In [ ]:
# Compléments métier: validation de contrat et granularité. Valeurs invalides = échec, pas de zéro.
keys={'candidats':['candidat_id'],'couverture_concurrence':['commune_code'],
 'mobilite_mesuree':['commune_code','annee'],'congestion':['commune_code'],
 'marche':['candidat_id','arret_id'],'faisabilite':['candidat_id'],
 'sinistres_citiz':['commune_code','annee']}
numeric={'mobilite_mesuree':['annee','nb_navetteurs','temps_moyen_min'],
 'congestion':['temps_pointe_min','temps_fluide_min','nb_observations'],
 'marche':['duree_marche_min'],'faisabilite':['cout_annuel_eur'],
 'sinistres_citiz':['annee','vehicules_jours','nb_vols','nb_degradations']}
for name,key in keys.items():
    t=B('input_'+name)
    if t.empty: continue
    if 'commune_code' in t:
        t['commune_code']=codes(t.commune_code)
        if (~t.commune_code.isin(com.commune_code)).any(): raise ValueError(name+': commune hors périmètre')
    for c in numeric.get(name,[]):
        t[c]=nnum(t[c])
        if t[c].isna().any() or (t[c]<0).any(): raise ValueError(name+': valeur non positive ou absente '+c)
    for c in [c for c in t if c.startswith('date_')]:
        t[c]=pd.to_datetime(t[c],errors='raise').dt.strftime('%Y-%m-%d')
    if name=='candidats':
        original=t.commune_code.copy(); t=assign(t)
        if t.commune_code.isna().any() or not (t.commune_code.astype(str).to_numpy()==original.astype(str).to_numpy()).all():
            raise ValueError('Candidat situé hors de sa commune déclarée')
    if name=='couverture_concurrence':
        t['inventaire_complet']=textnorm(t.inventaire_complet).isin(['true','1','oui'])
        age=(pd.Timestamp.today().normalize()-pd.to_datetime(t.date_observation)).dt.days
        t['inventaire_valide']=t.inventaire_complet & age.between(0,MAX_INVENTORY_AGE_DAYS)
    if name=='congestion':
        if (t.temps_fluide_min<=0).any() or (t.nb_observations<=0).any(): raise ValueError('Congestion: dénominateur nul')
        if (t.date_debut>t.date_fin).any(): raise ValueError('Congestion: dates inversées')
        t['indice_congestion']=(t.temps_pointe_min/t.temps_fluide_min-1).clip(lower=0)
    if name=='mobilite_mesuree' and (t.nb_navetteurs<=0).any(): raise ValueError('Nombre de navetteurs nul')
    if name=='sinistres_citiz' and (t.vehicules_jours<=0).any(): raise ValueError('Exposition nulle')
    if name=='faisabilite': t['stationnement_valide']=textnorm(t.stationnement_valide).isin(['true','1','oui'])
    assert_unique(t,key,name)
    write_table(SILVER,'input_'+name,t)
# Manifest Silver publié seulement si toutes les transformations ont réussi.
write_table(SILVER,'qualite_tables',pd.DataFrame(STATUS))
write_table(SILVER,'pipeline_run',pd.DataFrame([{'run_id':RUN_ID,'status':'complete','date_utc':dt.datetime.now(dt.timezone.utc).isoformat()}]))
